In [2]:
import os
from dotenv import load_dotenv
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline

# Load environment variables
load_dotenv()


DeprecatedPluginError: The `pinecone-plugin-inference` package has been deprecated. The features from that plugin have been incorporated into the main `pinecone` package with no need for additional plugins. Please remove the `pinecone-plugin-inference` package from your dependencies to ensure you have the most up-to-date version of these features.

In [ ]:


# Get API keys securely
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
HF_TOKEN = os.getenv("HUGGINGFACE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found. Check your .env file.")

if not HF_TOKEN:
    raise ValueError("HUGGINGFACE_API_KEY not found. Check your .env file.")


In [ ]:
def load_pdf_file(file_path):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    return documents

extracted_data = load_pdf_file("C:/Bits_hyd/doctor_pov/CuraMateDR/Data/tb.pdf")
print("Number of Pages Loaded:", len(extracted_data))


Number of Pages Loaded: 1874


In [ ]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

text_chunks = text_split(extracted_data)
print("Number of Text Chunks:", len(text_chunks))


Number of Text Chunks: 18109


In [ ]:
def get_hugging_face_embedding():
    return HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

embeddings = get_hugging_face_embedding()
print("Embeddings Loaded Successfully")


Embeddings Loaded Successfully


In [ ]:
print("Available Indexes:", pc.list_indexes())


Available Indexes: {'indexes': [{'deletion_protection': 'disabled',
              'dimension': 384,
              'host': 'medibot-vit7otl.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'medibot',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}}]}


In [ ]:
from pinecone import Pinecone

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY, environment="us-east-1")

# Connect to the existing index
index_name = "medibot"
index = pc.Index(index_name)

# Check index stats
print("Index is ready. Details:", index.describe_index_stats())


Index is ready. Details: {'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 36218}},
 'total_vector_count': 36218}


In [ ]:
from langchain_pinecone import PineconeVectorStore

# Connect to the existing index
docsearch = PineconeVectorStore.from_existing_index(
    index_name="medibot", 
    embedding=embeddings
)

# Create retriever
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 4})
print("Retriever Created Successfully")


Retriever Created Successfully


In [ ]:
def retrieve_text(question):
    retrieved_docs = retriever.invoke(question)
    retrieved_texts = [doc.page_content for doc in retrieved_docs]
    return "\n".join(retrieved_texts)

# Example
query = "What are the symptoms of tuberculosis?"
retrieved_text = retrieve_text(query)
print("Retrieved Context:", retrieved_text)


NameError: name 'retriever' is not defined